# Fake News Detection
A machine learning project to detect fake news articles using TF-IDF vectorization and Logistic Regression.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load and Explore Data

In [ ]:
# Load the datasets
fake_news = pd.read_csv("../Fake.csv")
true_news = pd.read_csv("../True.csv")

# Add label column (0 for Fake, 1 for Real)
fake_news['label'] = 0
true_news['label'] = 1

# Combine datasets
df = pd.concat([fake_news, true_news], ignore_index=True)

print(f"Total News Articles: {len(df)}")
print(f"Fake News: {len(fake_news)}")
print(f"Real News: {len(true_news)}")
print(f"\nDataset Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

In [ ]:
# Check data info
print("\nDataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nLabel Distribution:")
print(df['label'].value_counts())
print(f"\nPercentage of Fake vs Real:")
print(df['label'].value_counts(normalize=True) * 100)

## 2. Data Preprocessing

In [ ]:
# Combine text features to create a comprehensive article representation
df['text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

# Remove rows with no text
df = df[df['text'].str.strip() != '']

print(f"Dataset shape after preprocessing: {df.shape}")
print(f"Sample text:\n{df['text'].iloc[0][:200]}...")

## 3. Feature Engineering - TF-IDF Vectorization

In [ ]:
# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=5000, min_df=2, max_df=0.8, stop_words='english')

# Fit and transform the text data
X = vectorizer.fit_transform(df['text'])
y = df['label']

print(f"TF-IDF Matrix Shape: {X.shape}")
print(f"Number of Features: {X.shape[1]}")
print(f"Sparsity: {1 - (X.nnz / (X.shape[0] * X.shape[1]))} (~{(1 - (X.nnz / (X.shape[0] * X.shape[1]))) * 100:.2f}%)")

## 4. Train-Test Split

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")
print(f"Training set - Fake: {(y_train == 0).sum()}, Real: {(y_train == 1).sum()}")
print(f"Testing set - Fake: {(y_test == 0).sum()}, Real: {(y_test == 1).sum()}")

## 5. Model Training - Logistic Regression

In [ ]:
# Train Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"Model parameters:\n{model.get_params()}")

## 6. Model Evaluation

In [ ]:
# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Calculate metrics
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print("=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)
print(f"\nTraining Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")

print("\n" + "=" * 60)
print("DETAILED METRICS (Test Set)")
print("=" * 60)
print(f"Precision: {precision_score(y_test, y_pred_test):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_test):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_test):.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred_test, target_names=['Fake', 'Real']))

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(y_test, y_pred_test)
print(cm)

In [ ]:
# Visualize Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Fake', 'Real'], 
            yticklabels=['Fake', 'Real'])
plt.title('Confusion Matrix - Test Set')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# Plot accuracy comparison
fig, ax = plt.subplots(figsize=(8, 5))
models = ['Training', 'Testing']
accuracies = [train_accuracy, test_accuracy]
colors = ['#2ecc71', '#3498db']
bars = ax.bar(models, accuracies, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{acc:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.show()

## 7. Save Model and Vectorizer

In [ ]:
# Save the model and vectorizer for later use
joblib.dump(model, 'lr_model.jb')
joblib.dump(vectorizer, 'vectorizer.jb')

print("✓ Model saved as 'lr_model.jb'")
print("✓ Vectorizer saved as 'vectorizer.jb'")
print("\nThese files can now be used in the Streamlit app!")

## 8. Test Predictions on Sample News

In [ ]:
# Test with sample news articles
test_samples = [
    "Breaking News: Scientists Discover New Species in Amazon Rainforest",
    "SHOCKING: Celebrities Don't Want You To Know This One Weird Trick",
    "Report: Economic Growth Slows in Q4 2023",
    "FAKE: Government Secretly Testing 5G Mind Control Technology"
]

print("=" * 80)
print("SAMPLE PREDICTIONS")
print("=" * 80)

for i, sample in enumerate(test_samples, 1):
    # Vectorize the sample
    sample_vec = vectorizer.transform([sample])
    
    # Make prediction
    prediction = model.predict(sample_vec)[0]
    confidence = model.predict_proba(sample_vec)[0]
    
    label = "REAL" if prediction == 1 else "FAKE"
    
    print(f"\n[Sample {i}]")
    print(f"Text: {sample[:70]}...")
    print(f"Prediction: {label}")
    print(f"Confidence - Fake: {confidence[0]:.2%}, Real: {confidence[1]:.2%}")
    print("-" * 80)

## Summary

This notebook demonstrates a complete machine learning pipeline for **Fake News Detection**:

1. **Data Loading**: Loaded fake and real news datasets
2. **Preprocessing**: Combined title and text fields, removed empty entries
3. **Feature Engineering**: Used TF-IDF vectorization to convert text to numerical features
4. **Model Training**: Trained Logistic Regression on 80% of the data
5. **Evaluation**: Achieved high accuracy with detailed metrics
6. **Model Persistence**: Saved the model and vectorizer for production use
7. **Testing**: Tested predictions on sample news articles

The trained model is now ready to be deployed in the Streamlit web application (`app.py`) for real-time fake news detection!